# create

In [1]:
import pandas as pd
import string

from datetime import datetime, timedelta


## node zone

In [2]:
dtype_map = {
    "agg_pjm_ExternalNodeID": "string",
    "pjm_ExternalNodeID": "string",
    "agg_NodeName": "string",
    "NodeName": "string",
    "forecast_area": "string",
    "is_valid": "int8"
}

offset = 0
today_str = (datetime.today() - timedelta(days=offset)).strftime("%m%d") 

hourly_load = pd.read_csv(
    f"data/hourly_load_MW_by_distribution/hourly_load_MW_by_distribution_{today_str}.csv",
    skiprows=[1],
    dtype=dtype_map,
    parse_dates=["datetime_ending_ept", "insert_datetime"]
)

In [3]:
hourly_load.sort_values("datetime_ending_ept", inplace=True)

hourly_load = hourly_load[hourly_load["datetime_ending_ept"] >= "2026-03-12"].copy()

In [4]:
hourly_load.columns = (
    hourly_load.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

hourly_load = hourly_load.drop(columns=[
    "autokey",
    "insert_datetime"
], errors="ignore")

hourly_load["distribution_factor"] = (
    hourly_load["distribution_factor"]
    .astype(float)
    .round(8)
)

In [5]:
hourly_load = hourly_load.dropna(subset=[
    "datetime_ending_ept",
    "nodename",
    "distributed_load_mw"
])

hourly_load = hourly_load.drop_duplicates(
    subset=["datetime_ending_ept", "nodename"]
)

hourly_load.head()

,datetime_ending_ept,agg_pjm_externalnodeid,agg_nodename,pjm_externalnodeid,nodename,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,is_valid
8244966,2026-04-01,116013753,ATSI,2156114580,HILLS 69 KV LOAD1,0.00129,ATSI,6931.0,8.95485,1
8405482,2026-04-01,51301,PSEG,2041943359,PENNSNEC69 KV T3LD,0.00316,PSE&G/MIDATL,4341.0,13.70020,1
8405483,2026-04-01,51301,PSEG,48717,PIERSOAV230 KV T-1,0.00498,PSE&G/MIDATL,4341.0,21.63555,1
8405484,2026-04-01,51301,PSEG,48718,PIERSOAV230 KV T-2,0.00337,PSE&G/MIDATL,4341.0,14.64653,1
8405485,2026-04-01,51301,PSEG,1869174051,PLAINFIE69 KV T1,0.00065,PSE&G/MIDATL,4341.0,2.83467,1


In [6]:
hourly_load = hourly_load[["datetime_ending_ept", "agg_nodename", "pjm_externalnodeid", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].copy()

# Step 1: Replace nodename
hourly_load.loc[
    hourly_load["nodename"] == "SHAWVILL34 KV   3TX_LD",
    "nodename"
] = "SHAWVILL34 KV   CLR2"

# Step 2: Aggregate
hourly_load = (
    hourly_load.groupby(["datetime_ending_ept", "nodename"], as_index=False)
      .agg({
          "agg_nodename": "first",
          "pjm_externalnodeid": "first",
          "distribution_factor": "sum",
          "forecast_area": "first",
          "forecast_load_mw": "first",
          "distributed_load_mw": "sum"
      })
)


hourly_load["clean_nodename"] = (
    hourly_load["nodename"]
    .str.replace(r"[ _-]", "", regex=True)
    .str.upper()
    .str.replace(r"(LD|LOAD)(\d*)$", r"\2", regex=True)  # remove suffix
)

hourly_load.head()

,datetime_ending_ept,nodename,agg_nodename,pjm_externalnodeid,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,clean_nodename
0,2026-04-01,02ANGOLA138 KV TR1,ATSI,93353477,0.00078,ATSI,6931.0,5.39232,02ANGOLA138KVTR1
1,2026-04-01,02ANGOLA138 KV TR2,ATSI,93353479,0.00056,ATSI,6931.0,3.85364,02ANGOLA138KVTR2
2,2026-04-01,02ARMCO 138 KV TR1,ATSI,98369977,0.00128,ATSI,6931.0,8.85782,02ARMCO138KVTR1
3,2026-04-01,02ARMCO 138 KV TR2,ATSI,98369979,0.00128,ATSI,6931.0,8.85782,02ARMCO138KVTR2
4,2026-04-01,02ARMCO 138 KV TR3,ATSI,98369981,0.00128,ATSI,6931.0,8.85782,02ARMCO138KVTR3


## node bus

In [7]:
dtype_map = {
    "psse_bus_#": "string",
    "substation": "string",
    "nodename": "string",
    "nodekey": "string",
    "externalnodeid": "string",
    "lon": "float64",
    "lat": "float64",
}

pnode_to_bus = pd.read_csv('data/pnode_with_loc.csv')
pnode_to_bus.columns = (
    pnode_to_bus.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

pnode_to_bus = pnode_to_bus.astype({
    "externalnodeid": "Int64",
    "nodekey": "Int64"  # optional, if numeric
})

pnode_to_bus = pnode_to_bus.astype(dtype_map)

############################################################
nodename_mapping = {
    "BENJAMIN138 KV  BENDATLD": "BENJAMIN138 KV  DATCENLD",
    "960 ELGI4 KV    LAUXBLUE": "960 ELGI4 KV    BLUAUXLD",
    "960 ELGI4 KV    LAUXRED": "960 ELGI4 KV    REDAUXLD",
    "13 CRAWF138 KV  ATR57R04": "13 CRAWF138 KV  TR57R04",
    "13 CRAWF138 KV  ATR58R04": "13 CRAWF138 KV  TR58R04",
    "WARDAV  138 KV  COLONIAL": "WARDAV  138 KV  COLONILD",
    "PANTHER 69 KV   CRYPTOLD": "PANTHER 69 KV   CRYPT_LD",
    "SCRUBGRS13 KV   DATACEN1": "SCRUBGRS13 KV   DATCENT1",
    ############## from DAI
    "135 ELMH138 KV  TR76_12": "135 ELMH138 KV  TR76_34",
    "SHAWVILL34 KV   14 TX": "SHAWVILL34 KV   CLR2",
    "ENGLISHT35 KV   BK 5": "ENGLISHT34.5 KV A209",
    "ENGLISHT35 KV   BK 2": "ENGLISHT34.5 KV D82",
    "ENGLISHT35 KV   BK 1": "ENGLISHT34.5 KV G111",
    "SMITHBUR35 KV   LOAD1": "SMITHBUR34.5 KV TR3",
    "WIND JC 35 KV   BANK1": "WIND JC 34.5 KV G215_LD",
    "WIND JC 35 KV   BANK3": "WIND JC 34.5 KV H60",
    "WYCKOFF 115 KV  BANK3": "WYCKOFF 34.5 KV D82_LD",
    "CAPITALA138 KV  T2": "CAPITALA12 KV   T2",
    "BROSVILL138 KV  CITYDANV": "BROSVILL69 KV   CITYDANV",
    "POKAGON 138 KV  T2": "POKAGON 138 KV  T4"
}

pnode_to_bus["nodename"] = pnode_to_bus["nodename"].replace(nodename_mapping)
#############################################################

pnode_to_bus["clean_nodename"] = (
    pnode_to_bus["nodename"]
    .str.replace(r"[ _-]", "", regex=True)
    .str.upper()
    .str.replace(r"(LD|LOAD)(\d*)$", r"\2", regex=True)  # remove suffix
)

pnode_to_bus.head()

,psse_bus_#,substation,nodename,nodekey,externalnodeid,zonename,lon,lat,clean_nodename
0,10597,FOXBRKLN,FOXBRKLN230 KV LD_Y64,327047,2156118845,DOM DOMW,-78.047680,38.105092,FOXBRKLN230KVLDY64
1,12280,BLUEBELL,BLUEBELL69 KV KNOX2,327019,2156118844,FE FEOE,-81.083271,40.900624,BLUEBELL69KVKNOX2
2,2823,WOAKWOOD,WOAKWOOD69 KV T1,327166,2156118838,AEP AEPO,-84.338779,40.664493,WOAKWOOD69KVT1
3,1400,MUDDLAKE,MUDDLAKE34.5 KV AUX,327100,2156118830,AEP AEPI,-86.045797,41.956634,MUDDLAKE34.5KVAUX
4,9974,DAVESSTR,DAVESSTR34.5 KV LD3,327040,2156118822,DOM DOMN,-77.559931,38.855785,DAVESSTR34.5KV3


In [8]:
hourly_load.head()

,datetime_ending_ept,nodename,agg_nodename,pjm_externalnodeid,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,clean_nodename
0,2026-04-01,02ANGOLA138 KV TR1,ATSI,93353477,0.00078,ATSI,6931.0,5.39232,02ANGOLA138KVTR1
1,2026-04-01,02ANGOLA138 KV TR2,ATSI,93353479,0.00056,ATSI,6931.0,3.85364,02ANGOLA138KVTR2
2,2026-04-01,02ARMCO 138 KV TR1,ATSI,98369977,0.00128,ATSI,6931.0,8.85782,02ARMCO138KVTR1
3,2026-04-01,02ARMCO 138 KV TR2,ATSI,98369979,0.00128,ATSI,6931.0,8.85782,02ARMCO138KVTR2
4,2026-04-01,02ARMCO 138 KV TR3,ATSI,98369981,0.00128,ATSI,6931.0,8.85782,02ARMCO138KVTR3


In [9]:
# missing_nodenames = (
#     set(hourly_load["pjm_externalnodeid"]) 
#     - set(pnode_to_bus["externalnodeid"])
# )

# missing_nodenames

In [10]:
# missing_nodenames = (
#     set(hourly_load["clean_nodename"]) 
#     - set(pnode_to_bus["clean_nodename"])
# )

# missing_nodenames

In [11]:
# dup_keys = (
#     pnode_to_bus["clean_nodename"]
#     .value_counts()
#     .loc[lambda s: s > 1]
# )

# duplicate_rows = (
#     pnode_to_bus[pnode_to_bus["clean_nodename"].isin(dup_keys.index)]
#     .sort_values("clean_nodename")
# )

# duplicate_rows

In [12]:
cols = ["psse_bus_#", "substation", "nodename", "externalnodeid", "clean_nodename"]

pnode_map = pnode_to_bus[cols].copy()

hourly_load2 = hourly_load.copy()
hourly_load2["pjm_externalnodeid"] = hourly_load2["pjm_externalnodeid"].astype("Int64")
pnode_map["externalnodeid"] = pnode_map["externalnodeid"].astype("Int64")

# Optional but recommended: prevent duplicate name matches
pnode_map_by_name = pnode_map.drop_duplicates("clean_nodename", keep="first")

# 1) Match by externalnodeid
mapped = hourly_load2.merge(
    pnode_map,
    left_on="pjm_externalnodeid",
    right_on="externalnodeid",
    how="left",
    suffixes=("", "_bus")
)

miss_ext = mapped["psse_bus_#"].isna()

# 2) Match only missed rows by clean_nodename, preserving original row index
name_matches = (
    mapped.loc[miss_ext, ["clean_nodename"]]
    .reset_index(names="_row")
    .merge(
        pnode_map_by_name,
        on="clean_nodename",
        how="left",
        suffixes=("", "_bus")
    )
    .set_index("_row")
)

# Fill missed rows by index alignment
for col in cols:
    mapped.loc[name_matches.index, col] = (
        mapped.loc[name_matches.index, col]
        .combine_first(name_matches[col])
    )
    
hourly_load_mapped = mapped

unmapped_hourly_load = hourly_load_mapped[
    hourly_load_mapped["psse_bus_#"].isna()
].copy()

In [13]:
unmapped_hourly_load.head()

,datetime_ending_ept,nodename,agg_nodename,pjm_externalnodeid,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,clean_nodename,psse_bus_#,substation,nodename_bus,externalnodeid,clean_nodename_bus
4165,2026-04-01 00:00:00,GARDEN 69 KV T1,AEP,2156119597,0.00043,AEP,15108.0,6.46622,GARDEN69KVT1,<NA>,<NA>,<NA>,<NA>,<NA>
4166,2026-04-01 00:00:00,GARDEN 69 KV T4,AEP,2156118984,0.00021,AEP,15108.0,3.09714,GARDEN69KVT4,<NA>,<NA>,<NA>,<NA>,<NA>
14295,2026-04-01 01:00:00,GARDEN 69 KV T1,AEP,2156119597,0.00044,AEP,14305.0,6.29420,GARDEN69KVT1,<NA>,<NA>,<NA>,<NA>,<NA>
14296,2026-04-01 01:00:00,GARDEN 69 KV T4,AEP,2156118984,0.00022,AEP,14305.0,3.10418,GARDEN69KVT4,<NA>,<NA>,<NA>,<NA>,<NA>
24409,2026-04-01 02:00:00,GARDEN 69 KV T1,AEP,2156119597,0.00044,AEP,13979.0,6.10882,GARDEN69KVT1,<NA>,<NA>,<NA>,<NA>,<NA>


## Weave

In [14]:
unmapped_hourly_load['nodename'].unique()

<ArrowStringArray>
['GARDEN  69 KV   T1', 'GARDEN  69 KV   T4']
Length: 2, dtype: string

In [15]:
# nodename_mapping = {
#     "BENJAMIN138 KV  BENDATLD": "BENJAMIN138 KV  DATCENLD",
#     "960 ELGI4 KV    LAUXBLUE": "960 ELGI4 KV    BLUAUXLD",
#     "960 ELGI4 KV    LAUXRED": "960 ELGI4 KV    REDAUXLD",
#     "13 CRAWF138 KV  ATR57R04": "13 CRAWF138 KV  TR57R04",
#     "13 CRAWF138 KV  ATR58R04": "13 CRAWF138 KV  TR58R04",
#     "WARDAV  138 KV  COLONIAL": "WARDAV  138 KV  COLONILD",
#     "PANTHER 69 KV   CRYPTOLD": "PANTHER 69 KV   CRYPT_LD",
#     "SCRUBGRS13 KV   DATACEN1": "SCRUBGRS13 KV   DATCENT1",
#     ############## from DAI
#     "135 ELMH138 KV  TR76_12": "135 ELMH138 KV  TR76_34",
#     "SHAWVILL34 KV   14 TX": "SHAWVILL34 KV   CLR2",
#     "ENGLISHT35 KV   BK 5": "ENGLISHT34.5 KV A209",
#     "ENGLISHT35 KV   BK 2": "ENGLISHT34.5 KV D82",
#     "ENGLISHT35 KV   BK 1": "ENGLISHT34.5 KV G111",
#     "SMITHBUR35 KV   LOAD1": "SMITHBUR34.5 KV TR3",
#     "WIND JC 35 KV   BANK1": "WIND JC 34.5 KV G215_LD",
#     "WIND JC 35 KV   BANK3": "WIND JC 34.5 KV H60",
#     "WYCKOFF 115 KV  BANK3": "WYCKOFF 34.5 KV D82_LD",
#     "CAPITALA138 KV  T2": "CAPITALA12 KV   T2",
#     "BROSVILL138 KV  CITYDANV": "BROSVILL69 KV   CITYDANV",
#     "POKAGON 138 KV  T2": "POKAGON 138 KV  T4"
# }

# pnode_to_bus["nodename"] = pnode_to_bus["nodename"].replace(nodename_mapping)

## concatenation

In [15]:
mapped_hourly_load = hourly_load_mapped[
    hourly_load_mapped["psse_bus_#"].notna()
].copy()

hourly_load_bus_node = mapped_hourly_load[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw", "psse_bus_#", "substation"]].copy()


In [16]:
# hourly_load_bus_node = hourly_load.merge(pnode_to_bus[["psse_bus_#", "substation", "nodename"]], on = "nodename", how = "inner")
hourly_load_bus_node.head()

,datetime_ending_ept,agg_nodename,nodename,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,psse_bus_#,substation
0,2026-04-01,ATSI,02ANGOLA138 KV TR1,0.00078,ATSI,6931.0,5.39232,12897,02ANGOLA
1,2026-04-01,ATSI,02ANGOLA138 KV TR2,0.00056,ATSI,6931.0,3.85364,12897,02ANGOLA
2,2026-04-01,ATSI,02ARMCO 138 KV TR1,0.00128,ATSI,6931.0,8.85782,12206,02ARMCO
3,2026-04-01,ATSI,02ARMCO 138 KV TR2,0.00128,ATSI,6931.0,8.85782,12206,02ARMCO
4,2026-04-01,ATSI,02ARMCO 138 KV TR3,0.00128,ATSI,6931.0,8.85782,12206,02ARMCO


In [17]:
# drop unwanted column
df = hourly_load_bus_node.drop(columns=["forecast_area"]).copy()

df["psse_bus_#"] = df["psse_bus_#"].astype("int64")

# reorder columns
df = df[
    [
        "datetime_ending_ept",
        "psse_bus_#",
        "substation",
        "nodename",
        "distribution_factor",
        "distributed_load_mw",
        "agg_nodename",
        "forecast_load_mw"
    ]
]

# sort values
df = df.sort_values(
    by=[
        "datetime_ending_ept",   # primary
        "psse_bus_#",            # ascending
        "nodename"               # alphabetical
    ],
    ascending=[True, True, True]
).reset_index(drop=True)


In [18]:
df.tail()

,datetime_ending_ept,psse_bus_#,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw
10916071,2026-05-16,21992,YARDVILL,YARDVILL13 KV T1LD,0.00225,9.07609,PSEG,4041.0
10916072,2026-05-16,22014,SMECOMOR,SMECOMOR69 KV 6713,0.00197,4.88368,PEPCO,2474.0
10916073,2026-05-16,22038,HUNTSVIL,HUNTSVIL13 KV T1,0.00112,4.07147,PPL,3645.0
10916074,2026-05-16,22131,EWHITLEY,EWHITLEY34 KV LOAD1,0.00340,47.24980,AEP,13897.0
10916075,2026-05-16,22132,EWHITLEY,EWHITLEY34 KV LOAD2,0.00340,47.24980,AEP,13897.0


In [19]:

# rename
df = df.rename(columns={"psse_bus_#": "BusNum"})

# IDs: X1-X9, then XA-XZ
id_pool = [f"X{i}" for i in range(1, 10)] + [f"X{c}" for c in string.ascii_uppercase]

# assign ID within each BusNum by nodename order
node_id_map = (
    df[["BusNum", "nodename"]]
    .drop_duplicates()
    .sort_values(["BusNum", "nodename"])
    .assign(
        ID=lambda x: x.groupby("BusNum").cumcount().map(lambda i: id_pool[i])
    )
)

create_aux = df.merge(node_id_map, on=["BusNum", "nodename"], how="left")

# required AUX columns
create_aux["LabelAppend"] = create_aux["BusNum"].astype(str) + "-" + create_aux["ID"]
create_aux["LoadSMvar"] = float("0.0")
create_aux["LoadSMW"] = float("0.0")
create_aux["Status"] = "Closed"

create_aux.head()


,datetime_ending_ept,BusNum,substation,nodename,distribution_factor,distributed_load_mw,agg_nodename,forecast_load_mw,ID,LabelAppend,LoadSMvar,LoadSMW,Status
0,2026-04-01,4,ABSECON,ABSECON 69 KV LOAD2,0.00802,7.39905,AECO,922.0,X1,4-X1,0.0,0.0,Closed
1,2026-04-01,7,ATCO,ATCO 69 KV BUS1,0.01060,9.77412,AECO,922.0,X1,7-X1,0.0,0.0,Closed
2,2026-04-01,8,BARNEGAT,BARNEGAT69 KV T1,0.00912,8.40403,AECO,922.0,X1,8-X1,0.0,0.0,Closed
3,2026-04-01,9,BECKETT,BECKETT 69 KV T1,0.01952,17.99560,AECO,922.0,X1,9-X1,0.0,0.0,Closed
4,2026-04-01,9,BECKETT,BECKETT 69 KV T2,0.01050,9.68284,AECO,922.0,X2,9-X2,0.0,0.0,Closed


In [20]:
create_aux = create_aux[["LabelAppend", "BusNum", "ID", "LoadSMvar", "LoadSMW", "Status"]].drop_duplicates()
create_aux.head()

,LabelAppend,BusNum,ID,LoadSMvar,LoadSMW,Status
0,4-X1,4,X1,0.0,0.0,Closed
1,7-X1,7,X1,0.0,0.0,Closed
2,8-X1,8,X1,0.0,0.0,Closed
3,9-X1,9,X1,0.0,0.0,Closed
4,9-X2,9,X2,0.0,0.0,Closed


In [21]:
create_aux.shape

(10397, 6)

In [ ]:
base_case = pd.read_csv('data/PWD/base case.csv', skiprows = [0])
base_case = base_case[["Label", "BusNum", "AreaName", "LoadID", "LoadMW",  "LoadStatus"]]
base_case.rename(columns={"Label": "LabelAppend", "LoadID": "ID", "LoadMW": "LoadSMW", "LoadStatus": "Status"}, inplace=True)
base_case["ID"] = base_case["ID"].astype(str)
base_case = base_case[base_case["AreaName"] != "PJM"].copy()
base_case.head()

,LabelAppend,BusNum,AreaName,ID,LoadSMW,Status
0,159-1,159,AECI,1,0.0,Closed
1,203-1,203,AECI,1,0.0,Closed
2,242-3,242,AECI,3,0.0,Closed
3,172-1,172,AECI,1,0.0,Closed
4,247-2,247,AECI,2,0.0,Closed


In [23]:
base_case.shape

(19908, 6)

In [24]:
base_case[base_case["AreaName"] == "PJM"].shape

(11591, 6)

In [25]:
# Find BusNums already in create_aux
existing_busnums = set(create_aux["BusNum"])

# Filter base_case for rows NOT in create_aux
missing_rows = base_case[~base_case["BusNum"].isin(existing_busnums)].copy()

In [26]:

# Align columns (important)
missing_rows = missing_rows.reindex(columns=create_aux.columns)

# Append
create_aux_updated = pd.concat([create_aux, missing_rows], ignore_index=True)

create_aux_updated.head()

,LabelAppend,BusNum,ID,LoadSMvar,LoadSMW,Status
0,4-X1,4,X1,0.0,0.0,Closed
1,7-X1,7,X1,0.0,0.0,Closed
2,8-X1,8,X1,0.0,0.0,Closed
3,9-X1,9,X1,0.0,0.0,Closed
4,9-X2,9,X2,0.0,0.0,Closed


In [27]:
# build lines (fast)
lines = (
    '"' 
    + create_aux_updated["LabelAppend"].fillna("").astype(str)
    + '" '
    + create_aux_updated["BusNum"].fillna("").astype(str)
    + ' "'
    + create_aux_updated["ID"].fillna("").astype(str)
    + '" '
    + create_aux_updated["LoadSMvar"].fillna(0).astype(str)
    + " "
    + create_aux_updated["LoadSMW"].fillna(0).astype(str)
    + ' "'
    + create_aux_updated["Status"].fillna("").astype(str)
    + '"'
).astype(str).tolist()

# assemble full AUX text
aux_text = (
    "Load (LabelAppend,BusNum,ID,SMvar,SMW,Status)\n"
    "{\n"
    + "\n".join(lines)
    + "\n}"
)

date_str = pd.Timestamp.today().normalize().strftime("%Y%m%d")

# write to file
with open(f"data/PWD/load_create/load_create_{date_str}.aux", "w", encoding="utf-8") as f:
    f.write(aux_text)

In [28]:
create_aux_updated = create_aux_updated.rename(columns={
    "LabelAppend": "LabelAppend",
    "BusNum": "BusNum",
    "ID": "ID",
    "LoadSMvar": "SMvar",
    "LoadSMW": "SMW",
    "Status": "Status"
})

create_aux_updated["LabelAppend"] = ""

create_aux_updated = create_aux_updated.dropna(how="all")

date_str = pd.Timestamp.today().normalize().strftime("%Y%m%d")

csv_path = f"data/PWD/load_create/load_create_{date_str}.csv"

with open(csv_path, "w", encoding="utf-8") as f:

    # first line
    f.write("Load\n")

    # actual table
    create_aux_updated.to_csv(
        f,
        index=False,
        lineterminator="\n"
    )

print(f"CSV written to: {csv_path}")

CSV written to: data/PWD/load_create/load_create_20260515.csv
